# Silver-to-Gold: Carregando o Data Warehouse

## Objetivos

- Ler os dados da camada Silver (tabela `escolas`).
- Popular as tabelas de dimensão.
- Popular a tabela fato (`fato_escola`) e a tabela bridge (`bridge_escola_etapa`).
- Garantir a idempotência do processo de carga.

#### Configuração do Spark

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.appName("Silver to Gold ETL")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate()
    )

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/12 17:37:06 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.1.41 instead (on interface wlp63s0)
25/11/12 17:37:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/lucas/fga/bancos2/grupo-18-bancos-2/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/lucas/.ivy2.5.2/cache
The jars for the packages stored in: /home/lucas/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f6b5d5c8-bad3-4922-b0dd-5b072693d5e4;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 65ms :: artifacts dl 2ms
	:: modules in use:
	org.checkerframewo

#### Configuração do Banco de Dados

In [2]:
postgres_host = os.getenv("POSTGRES_HOST", "localhost")
postgres_port = os.getenv("POSTGRES_PORT", "5432")
postgres_db = os.getenv("POSTGRES_DB", "inep_db")
postgres_user = os.getenv("POSTGRES_USER", "inep")
postgres_password = os.getenv("POSTGRES_PASSWORD", "inep")
silver_table = os.getenv("POSTGRES_TABLE", "escolas")

jdbc_url = f"jdbc:postgresql://{postgres_host}:{postgres_port}/{postgres_db}"
jdbc_properties = {
    "user": postgres_user,
    "password": postgres_password,
    "driver": "org.postgresql.Driver"
}

print(f"Conectando ao banco de dados: {jdbc_url}")

Conectando ao banco de dados: jdbc:postgresql://localhost:5432/inep_db


#### Carregar Dados da Camada Silver

In [3]:
print(f"Lendo dados da tabela silver: '{silver_table}'")

df_silver = (
    spark.read.jdbc(url=jdbc_url, table=silver_table, properties=jdbc_properties)
    .withColumn("is_rural", F.col("is_rural").cast("boolean"))
    .withColumn("is_publica", F.col("is_publica").cast("boolean"))
)

df_silver.cache()

print(f"Total de registros lidos: {df_silver.count():,}")
df_silver.printSchema()

Lendo dados da tabela silver: 'escolas'


Total de registros lidos: 156,423
root
 |-- codigo_inep: string (nullable = true)
 |-- nome_escola: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- localizacao: string (nullable = true)
 |-- is_rural: boolean (nullable = true)
 |-- dependencia_administrativa: string (nullable = true)
 |-- is_publica: boolean (nullable = true)
 |-- porte_escola: string (nullable = true)
 |-- porte_numerico: integer (nullable = true)
 |-- etapas_modalidades: string (nullable = true)
 |-- num_etapas: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- restricao_atendimento: string (nullable = true)



## 1. Popular Tabelas de Dimensão

In [4]:
from psycopg2 import connect, sql

def _qualified_table(table_name: str):
    """Retorna o identificador qualificado para schema.tabela."""
    if "." in table_name:
        schema, table = table_name.split(".", 1)
        return sql.SQL("{}.{}").format(sql.Identifier(schema), sql.Identifier(table))
    return sql.Identifier(table_name)


def write_rows_individually(df, table_name, truncate=True, cascade=False):
    """
    Insere os registros de um DataFrame linha a linha na tabela alvo.
    Em caso de falha, exibe o índice e o conteúdo da linha problemática.
    """
    columns = df.columns
    if not columns:
        print(f"Nenhuma coluna disponível para inserir em '{table_name}'.")
        return

    conn = connect(
        host=postgres_host,
        port=postgres_port,
        dbname=postgres_db,
        user=postgres_user,
        password=postgres_password
    )
    conn.autocommit = False
    cursor = conn.cursor()

    qualified_table = _qualified_table(table_name)
    column_identifiers = sql.SQL(", ").join(sql.Identifier(col) for col in columns)
    placeholders = sql.SQL(", ").join(sql.Placeholder() for _ in columns)
    insert_stmt = sql.SQL("INSERT INTO {} ({}) VALUES ({})").format(
        qualified_table, column_identifiers, placeholders
    )

    try:
        if truncate:
            truncate_stmt = sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY").format(
                qualified_table
            )
            if cascade:
                truncate_stmt += sql.SQL(" CASCADE")
            cursor.execute(truncate_stmt)
            conn.commit()
            print(f"Tabela '{table_name}' truncada antes da nova carga.")

        row_count = 0
        for row_count, row in enumerate(df.toLocalIterator(), start=1):
            values = [row[col] for col in columns]
            try:
                cursor.execute(insert_stmt, values)
                conn.commit()
            except Exception as err:
                conn.rollback()
                print(
                    f"Falha ao inserir linha {row_count} em '{table_name}': {row.asDict()}"
                )
                raise err

        print(f"Tabela '{table_name}' carregada com {row_count:,} registros.")
    finally:
        cursor.close()
        conn.close()

def load_dimension(
    df, table_name, select_cols, distinct=True, rename_map=None, drop_nulls=False
):
    """Função para extrair, transformar e carregar uma tabela de dimensão."""
    print(f"Processando dimensão: {table_name}")
    dim_df = df.select(*select_cols)

    if distinct:
        dim_df = dim_df.distinct()

    if rename_map:
        for old_name, new_name in rename_map.items():
            dim_df = dim_df.withColumnRenamed(old_name, new_name)

    if drop_nulls:
        dim_df = dim_df.dropna(subset=dim_df.columns)

    write_rows_individually(dim_df, table_name, cascade=True)

    # Retorna o DF lido do DW para obter os SKs gerados
    return spark.read.jdbc(url=jdbc_url, table=table_name, properties=jdbc_properties)

#### 1.1. dim_escola

In [5]:
dim_escola_sk = load_dimension(
    df=df_silver,
    table_name="dim_escola",
    select_cols=["codigo_inep", "nome_escola"]
)

Processando dimensão: dim_escola
Tabela 'dim_escola' truncada antes da nova carga.
Tabela 'dim_escola' carregada com 156,423 registros.


#### 1.2. dim_localidade

In [6]:
df_localidade = df_silver.withColumn("is_rural", F.col("is_rural").cast("boolean"))

dim_localidade_sk = load_dimension(
    df=df_localidade,
    table_name="dim_localidade",
    select_cols=["uf", "municipio", "regiao", "localizacao", "is_rural", "latitude", "longitude"]
)

Processando dimensão: dim_localidade
Tabela 'dim_localidade' truncada antes da nova carga.
Tabela 'dim_localidade' carregada com 156,184 registros.


#### 1.3. dim_dependencia

In [7]:
df_dependencia = df_silver.withColumn("is_publica", F.col("is_publica").cast("boolean"))

dim_dependencia_sk = load_dimension(
    df=df_dependencia,
    table_name="dim_dependencia",
    select_cols=["dependencia_administrativa", "is_publica"],
    rename_map={"dependencia_administrativa": "dependencia_adm"}
)

Processando dimensão: dim_dependencia
Tabela 'dim_dependencia' truncada antes da nova carga.
Tabela 'dim_dependencia' carregada com 4 registros.


#### 1.4. dim_porte

In [8]:
dim_porte_sk = load_dimension(
    df=df_silver,
    table_name="dim_porte",
    select_cols=["porte_escola", "porte_numerico"],
    drop_nulls=True
)

Processando dimensão: dim_porte
Tabela 'dim_porte' truncada antes da nova carga.
Tabela 'dim_porte' carregada com 5 registros.


#### 1.5. dim_restricao_atendimento

In [9]:
dim_restricao_sk = load_dimension(
    df=df_silver,
    table_name="dim_restricao_atendimento",
    select_cols=["restricao_atendimento"],
    rename_map={"restricao_atendimento": "restricao_desc"}
)

Processando dimensão: dim_restricao_atendimento
Tabela 'dim_restricao_atendimento' truncada antes da nova carga.
Tabela 'dim_restricao_atendimento' carregada com 5 registros.


#### 1.6. dim_etapa

In [10]:
df_etapas = df_silver.select(
    F.explode(F.split(F.col("etapas_modalidades"), ",")).alias("etapa")
).withColumn("etapa", F.trim(F.col("etapa"))).distinct()

dim_etapa_sk = load_dimension(
    df=df_etapas,
    table_name="dim_etapa",
    select_cols=["etapa"],
    distinct=False
)

Processando dimensão: dim_etapa
Tabela 'dim_etapa' truncada antes da nova carga.
Tabela 'dim_etapa' carregada com 5 registros.


## 2. Popular Tabela Fato e Bridge

In [11]:
df_fato_base = (
    df_silver.join(dim_escola_sk, "codigo_inep", "inner")
    .join(dim_localidade_sk, ["uf", "municipio", "regiao", "localizacao", "is_rural", "latitude", "longitude"], "inner")
    .join(dim_dependencia_sk, F.col("dependencia_administrativa") == F.col("dependencia_adm"), "inner")
    .join(dim_porte_sk, ["porte_escola", "porte_numerico"], "left")
    .join(dim_restricao_sk, F.col("restricao_atendimento") == F.col("restricao_desc"), "left")
)

print("Joins com dimensões concluídos.")

Joins com dimensões concluídos.


In [12]:
df_fato = df_fato_base.select(
    "sk_escola",
    "sk_localidade",
    "sk_dependencia",
    "sk_porte",
    "sk_restricao"
).distinct()

write_rows_individually(df_fato, "fato_escola", cascade=True)

print(f"Tabela 'fato_escola' carregada com {df_fato.count():,} registros.")

Tabela 'fato_escola' truncada antes da nova carga.


Tabela 'fato_escola' carregada com 156,423 registros.


Tabela 'fato_escola' carregada com 156,423 registros.


In [13]:
df_bridge_base = df_silver.select(
    "codigo_inep",
    F.explode(F.split(F.col("etapas_modalidades"), ",")).alias("etapa")
).withColumn("etapa", F.trim(F.col("etapa")))

df_bridge = (
    df_bridge_base.join(dim_escola_sk, "codigo_inep", "inner")
    .join(dim_etapa_sk, "etapa", "inner")
    .select("sk_escola", "sk_etapa")
    .distinct()
)

write_rows_individually(df_bridge, "bridge_escola_etapa", cascade=True)

print(f"Tabela 'bridge_escola_etapa' carregada com {df_bridge.count():,} registros.")

Tabela 'bridge_escola_etapa' truncada antes da nova carga.
Tabela 'bridge_escola_etapa' carregada com 248,109 registros.
Tabela 'bridge_escola_etapa' carregada com 248,109 registros.


In [14]:
df_silver.unpersist()
spark.stop()
print("Sessão Spark finalizada.")

Sessão Spark finalizada.
